<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/main/code/stable-baselines/cartpole-sb3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQN and PPO for _cartpole_ with Stable-Baselines3

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
This work is licensed under the [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Based on:
> A. Raffin et al. (2021) Stable-Baselines3: Reliable Reinforcement Learning Implementations. _Journal of Machine Learning Research_ 22(268):1-8. Available on [GitHub](https://github.com/DLR-RM/stable-baselines3).

> A. Raffin (2020) RL Baselines3 Zoo. Available on [GitHub](https://github.com/DLR-RM/rl-baselines3-zoo).

In the notebooks of the [`from-scratch`](https://github.com/jgromero/drl-csic/tree/main/code/from-scratch) folder we implemented DQN and REINFORCE ourselves. Here we solve the same problem with [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3), a library of reliable, tested implementations of the most common DRL algorithms built on PyTorch. With SB3, the agent, the neural network, the replay buffer and the training loop are already implemented: we only have to choose an algorithm, set its hyperparameters and call `learn()`.

We will train two agents:

*   [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html): the same value-based algorithm of [cartpole-dqn.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-dqn.ipynb).
*   [PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html): a policy-gradient (actor-critic) algorithm that improves on the ideas of REINFORCE ([cartpole-reinforce.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-reinforce.ipynb)).

## Configuration

In [ ]:
%pip install "gymnasium[classic-control]>=1.1" "stable-baselines3>=2.6" tqdm rich

In [ ]:
# set render to 'rgb_array' (change to 'human' for local display-enabled runtime)
render_mode = "rgb_array"

## Environment

We will be using [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), a simple control problem.

<img src="https://github.com/jgromero/drl-csic/blob/main/img/cartpole.gif?raw=true"/>

The [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment is considered solved when the agent obtains an average score of at least $475$ points over 100 consecutive episodes. Episodes are truncated at $500$ steps, which is therefore the maximum score.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

SB3 needs the environment to be wrapped with a [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) to keep track of the score and length of each episode. We will use these values to plot the learning curves.

In [ ]:
SOLVED_SCORE = 475.0   # average score (last 100 episodes) to consider CartPole-v1 solved

def plot_scores(env, title):
    """Plot the score of every training episode and the moving average of the last 100.

    Params
    ======
        env: environment (or vectorized environment) wrapped with Monitor
        title (str): plot title
    """
    # a vectorized environment contains several Monitor-wrapped copies of the environment
    monitors = [env] if isinstance(env, Monitor) else env.envs
    scores = np.concatenate([m.get_episode_rewards() for m in monitors])
    ends = np.concatenate([np.cumsum(m.get_episode_lengths()) for m in monitors])
    scores = scores[np.argsort(ends)]   # sort episodes by the (per-env) time step when they finished

    avg = [np.mean(scores[max(0, i-99):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label='Score')
    plt.plot(np.arange(len(scores)), avg, label='Average score (last 100)')
    plt.axhline(SOLVED_SCORE, color='gray', linestyle='--', label='Solved')
    plt.ylabel('Score')
    plt.xlabel('Episode #')
    plt.title(title)
    plt.legend()
    plt.show()

## DQN algorithm

The [`DQN`](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) class includes the Q-network, the target network, the replay buffer and the $\epsilon$-greedy exploration strategy. Compare its parameters with those of our own implementation in [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py):

| SB3 parameter | Meaning | `dqn_agent.py` |
|---|---|---|
| `buffer_size` | replay buffer size | `BUFFER_SIZE` |
| `batch_size` | minibatch size | `BATCH_SIZE` |
| `gamma` | discount factor | `GAMMA` |
| `learning_rate` | learning rate | `LR` |
| `train_freq` | how often to update the network (steps) | `UPDATE_EVERY` |
| `target_update_interval` | how often to update the target network (steps) | `TAU` (soft update) |
| `exploration_*` | $\epsilon$-greedy schedule | `eps_start`, `eps_end`, `eps_decay` |
| `policy_kwargs` | architecture of the Q-network | [model.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/model.py) |

We use the hyperparameters tuned for CartPole-v1 in the [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml). Instead of a number of episodes, SB3 trains for a given number of time steps (`total_timesteps`).

In [ ]:
env_dqn = Monitor(gym.make("CartPole-v1"))

model_dqn = DQN(
    "MlpPolicy",                        # Q-network: multilayer perceptron
    env_dqn,
    learning_rate=2.3e-3,
    batch_size=64,
    buffer_size=100_000,
    learning_starts=1000,               # steps of random actions before learning starts
    gamma=0.99,
    target_update_interval=10,
    train_freq=256,                     # update the network every 256 steps...
    gradient_steps=128,                 # ...with 128 minibatches
    exploration_fraction=0.16,          # epsilon decays linearly during the first 16% of training...
    exploration_final_eps=0.04,         # ...to this value
    policy_kwargs=dict(net_arch=[256, 256]),
    seed=0,
    verbose=0,
)

model_dqn.learn(total_timesteps=50_000, progress_bar=True)
model_dqn.save("cartpole-sb3-dqn")   # save trained agent

plot_scores(env_dqn, 'DQN')

The score shown during training is obtained with the $\epsilon$-greedy policy, i.e., with some random actions. To assess the learned policy, we evaluate the agent with the greedy policy (`deterministic=True`) for 100 episodes.

In [ ]:
mean_score, std_score = evaluate_policy(model_dqn, Monitor(gym.make("CartPole-v1")), n_eval_episodes=100, deterministic=True)
print('DQN\tAverage score (100 episodes): {:.2f} +/- {:.2f}'.format(mean_score, std_score))

## PPO algorithm

[Proximal Policy Optimization](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html) (PPO) is a policy-gradient method, like REINFORCE, with several improvements:

*   It is an _actor-critic_ method: besides the policy network (actor), it learns a value network (critic) that is used to reduce the variance of the gradient estimate (instead of the normalized return $G_t$ that we used in REINFORCE).
*   It limits (_clips_) the size of each policy update, so that the new policy does not move too far from the previous one. This makes it possible to reuse each batch of experience for several gradient steps (`n_epochs`).
*   It collects experience from several copies of the environment in parallel (`n_envs`), which SB3 builds with [`make_vec_env`](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html).

Again, we use the hyperparameters of the [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/ppo.yml). Note that we only need to change the class of the model: the rest of the code is the same.

In [ ]:
env_ppo = make_vec_env("CartPole-v1", n_envs=8, seed=0)   # 8 parallel environments, each one wrapped with Monitor

model_ppo = PPO(
    "MlpPolicy",                        # actor and critic networks: multilayer perceptrons
    env_ppo,
    learning_rate=1e-3,
    n_steps=32,                         # steps collected from each environment before each update
    batch_size=256,
    n_epochs=20,                        # gradient steps (epochs) with each batch of experience
    gamma=0.98,
    gae_lambda=0.8,
    clip_range=0.2,                     # maximum change of the policy in each update
    ent_coef=0.0,
    seed=0,
    verbose=0,
)

model_ppo.learn(total_timesteps=100_000, progress_bar=True)
model_ppo.save("cartpole-sb3-ppo")   # save trained agent

plot_scores(env_ppo, 'PPO')

In [ ]:
mean_score, std_score = evaluate_policy(model_ppo, Monitor(gym.make("CartPole-v1")), n_eval_episodes=100, deterministic=True)
print('PPO\tAverage score (100 episodes): {:.2f} +/- {:.2f}'.format(mean_score, std_score))

## Visualize trained agent

We can afterwards load the trained agents and visualize how they act on the environment. (Local run is required for 2D rendering, not Google Colaboratory.)

In [ ]:
# load trained agent from `cartpole-sb3-ppo.zip` (use DQN.load("cartpole-sb3-dqn") for the DQN agent)
model_load = PPO.load("cartpole-sb3-ppo")

for i in range(3):
    state, _ = env.reset()
    for j in range(500):
        action, _ = model_load.predict(state, deterministic=True)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()